In [1]:
import numpy as np 
import soundfile as sf
import librosa
import pandas as pd 
from pathlib import Path
import pickle 
import IPython.display as ipd
import sys 
sys.path.append('../../datasets/spatial_audio_pipeline/spatial_audio_util/')
import itertools
import util_audio 
import soxr
import h5py

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/scipy/__init__.py:138: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4)
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion} is required for this version of "


# Make stimuli for 2024 azimuth "spotlight broadening" experiment

## Wanted Conditions:
### SNRs: 0dB

### Target azimuths: 0, 90 
### Distractor azimuths: 0, 10, 20, 30, 40, 60, 90 



Will be using cues, foregrounds, and english distractors from:
`/om/user/imgriff/datasets/human_word_rec_SWC_2024/full_cue_target_distractor_df_w_meta_transcripts.pdpkl`

This manifest and
stimuli will be moved to `/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024`


##### NOTE: Sounds actually cut and written using `src/get_swc_mono_expmnt_stim_2024`. This notebook prepares the manifest for that script.

In [2]:
stim_out_path = Path('/om/user/imgriff/datasets/human_azim_spotlight_SWC_2024')
stim_out_path.mkdir(exist_ok=True, parents=True)
manifest = pd.read_pickle('/om/user/imgriff/datasets/human_word_rec_SWC_2024/human_cue_target_distractor_df_w_meta_transcripts_w_f0.pdpkl')
manifest.shape

(976, 46)

In [20]:
### Init spatial conditions

target_azims = [0, 90, 270] # Remember: 90 is left and 270 is right in array simulation
distractor_deltas = np.hstack([np.arange(0,41,10), [60,90]])

# rotations are inverted - counter clockwise is positve and clockwise is negative

left_conditions = list(itertools.product([90], -distractor_deltas)) # negative delta to rotate clockwise toward midline 
right_conditions = list(itertools.product([270], distractor_deltas)) # positive delta to rotate counter clockwise toward midline
center_conditions = list(itertools.product([0], np.hstack([distractor_deltas, -distractor_deltas])))



all_conditions = sorted(set(left_conditions + right_conditions + center_conditions)) # drop duplicates 

def get_distractor_azim(target_azim, delta):
    dist_azim = target_azim + delta
    return dist_azim % 360

## Make condition dict 
condition_dict = {ix: {'target_azim':cond[0],
                       "dist_azim_delta":cond[1],
                       "distractor_azim":get_distractor_azim(cond[0], cond[1])}
                        for ix, cond in enumerate(all_conditions)}

out_name = stim_out_path/"human_azim_spotlight_cond_map.pkl"
# write condition dict as pickle 
with open(out_name, 'wb') as f:
    pickle.dump(condition_dict, f)

In [19]:
condition_dict

{0: {'target_azim': 0, 'dist_azim_delta': -90, 'distractor_azim': 270},
 1: {'target_azim': 0, 'dist_azim_delta': -60, 'distractor_azim': 300},
 2: {'target_azim': 0, 'dist_azim_delta': -40, 'distractor_azim': 320},
 3: {'target_azim': 0, 'dist_azim_delta': -30, 'distractor_azim': 330},
 4: {'target_azim': 0, 'dist_azim_delta': -20, 'distractor_azim': 340},
 5: {'target_azim': 0, 'dist_azim_delta': -10, 'distractor_azim': 350},
 6: {'target_azim': 0, 'dist_azim_delta': 0, 'distractor_azim': 0},
 7: {'target_azim': 0, 'dist_azim_delta': 10, 'distractor_azim': 10},
 8: {'target_azim': 0, 'dist_azim_delta': 20, 'distractor_azim': 20},
 9: {'target_azim': 0, 'dist_azim_delta': 30, 'distractor_azim': 30},
 10: {'target_azim': 0, 'dist_azim_delta': 40, 'distractor_azim': 40},
 11: {'target_azim': 0, 'dist_azim_delta': 60, 'distractor_azim': 60},
 12: {'target_azim': 0, 'dist_azim_delta': 90, 'distractor_azim': 90},
 13: {'target_azim': 90, 'dist_azim_delta': -90, 'distractor_azim': 0},
 14: 

In [12]:
len(condition_dict)

27

In [21]:
condition_dict

{0: {'target_azim': 0, 'dist_azim_delta': -90, 'distractor_azim': 270},
 1: {'target_azim': 0, 'dist_azim_delta': -60, 'distractor_azim': 300},
 2: {'target_azim': 0, 'dist_azim_delta': -40, 'distractor_azim': 320},
 3: {'target_azim': 0, 'dist_azim_delta': -30, 'distractor_azim': 330},
 4: {'target_azim': 0, 'dist_azim_delta': -20, 'distractor_azim': 340},
 5: {'target_azim': 0, 'dist_azim_delta': -10, 'distractor_azim': 350},
 6: {'target_azim': 0, 'dist_azim_delta': 0, 'distractor_azim': 0},
 7: {'target_azim': 0, 'dist_azim_delta': 10, 'distractor_azim': 10},
 8: {'target_azim': 0, 'dist_azim_delta': 20, 'distractor_azim': 20},
 9: {'target_azim': 0, 'dist_azim_delta': 30, 'distractor_azim': 30},
 10: {'target_azim': 0, 'dist_azim_delta': 40, 'distractor_azim': 40},
 11: {'target_azim': 0, 'dist_azim_delta': 60, 'distractor_azim': 60},
 12: {'target_azim': 0, 'dist_azim_delta': 90, 'distractor_azim': 90},
 13: {'target_azim': 90, 'dist_azim_delta': -90, 'distractor_azim': 0},
 14: 

In [29]:
condition_dict[0]

{'target_azim': 0, 'dist_azim_delta': -90}

#### Setup experiment word list

In [ ]:
stim_out_path

: 

In [26]:
# Make word key 
import json
target_words = np.sort(manifest['word'].unique())
#enumerate words as dict 
target_word_dict = dict(enumerate(target_words))
# save as pickle
out_name = stim_out_path / "human_azim_spotlight_word_key.pkl"
with open(out_name, 'wb') as f:
    pickle.dump(target_word_dict, f)

words = list(target_word_dict.values())
# save as json 
out_name = stim_out_path /  "human_azim_spotlight_word_key.js"
with open(out_name, 'w') as f:
    json.dump({"dictionary":words}, f)


In [5]:
## Map words to experiment word list ix 
word_to_exp_ix = {word: ix for ix, word in target_word_dict.items()}


#### Pre-cut down to one distractor per example.

In [6]:
manifest.columns

Index(['manifest_ix', 'orig_df_ix', 'client_id', 'corpus', 'gender',
       'gender_int', 'sr', 'word', 'cue_client_id', 'cue_corpus', 'cue_gender',
       'cue_gender_int', 'cue_sr', 'cue_total_file_duration_in_s', 'cue_word',
       'sex_cond', 'excerpt_src_fn', 'excerpt_cue_src_fn',
       'excerpt_same_sex_distractor_1_src_fn',
       'excerpt_same_sex_distractor_2_src_fn', 'same_sex_distractor_1_src_fn',
       'same_sex_distractor_2_src_fn', 'target_transcripts',
       'same_sex_distractor_1_transcripts',
       'same_sex_distractor_2_transcripts', 'same_sex_dist_1_word',
       'same_sex_dist_2_word', 'excerpt_diff_sex_distractor_1_src_fn',
       'excerpt_diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_src_fn',
       'diff_sex_distractor_2_src_fn', 'diff_sex_distractor_1_transcripts',
       'diff_sex_distractor_2_transcripts', 'diff_sex_dist_1_word',
       'diff_sex_dist_2_word', 'same_sex_zh_distractor_full_path',
       'diff_sex_zh_distractor_full_path', 'same_sex_

In [7]:
np.random.seed(0)
dist_cond = np.random.choice(['same', 'diff'], size=manifest.shape[0])

dist_word_and_tscripts = [manifest.loc[ix, [f"excerpt_{cond}_sex_distractor_1_src_fn", f"{cond}_sex_dist_1_word", f"{cond}_sex_distractor_1_transcripts"]].values for ix, cond in enumerate(dist_cond)]

human_exp_df = manifest.copy()
human_exp_df['excerpt_distractor_src_fn'], human_exp_df['distractor_word'], human_exp_df['distractor_transcripts'] = zip(*dist_word_and_tscripts)
human_exp_df['sex_cond'] = dist_cond

## drop columns with same or diff in them, or full_path in them 
human_exp_df = human_exp_df.drop(columns=[col for col in human_exp_df.columns if 'same' in col or 'diff' in col or "full_path" in col])

In [8]:
human_exp_df["exp_word_ix"] = human_exp_df['word'].replace(word_to_exp_ix)

In [9]:
import pickle 

cv_word_2_class = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
cv_class_2_word = {v:k for k,v in cv_word_2_class.items()}

In [10]:
### Add cv word int to manifest 
human_exp_df["word_int_label"] = human_exp_df['word'].replace(cv_word_2_class)

## Demo spatialization

### Will run using min-reverb room 

In [11]:
test_IR_manifest_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
room_ix = 0
test_IR_manifest_path = test_IR_manifest_dir / "manifest_brir.pdpkl"
h5_fn = test_IR_manifest_dir / f"room{room_ix:04}.hdf5"
new_room_manifest = pd.read_pickle(test_IR_manifest_path)
only14_manifest = new_room_manifest[(new_room_manifest['index_room'] == room_ix)  & (new_room_manifest['src_dist'] == 1.4)]


In [12]:
def get_brir(azim=None, elev=None, coords=None, h5_fn=None, IR_df=None, out_sr=44_100):
    if coords is not None:
        azim, elev = coords
    df_row = IR_df[(IR_df['src_azim'] == azim) & (IR_df['src_elev'] == elev)]
    brir_ix = df_row['index_brir'].values[0]
    sr_src = df_row['sr'].values[0]
    with h5py.File(h5_fn, 'r') as f:
        brir = f['brir'][brir_ix]
    if out_sr != sr_src:
        brir = soxr.resample(brir.astype(np.float32), sr_src, out_sr)
    return brir


In [13]:
target_brir = get_brir(coords=(0,0), h5_fn=h5_fn, IR_df=only14_manifest)
dist_brir = get_brir(coords=(270,0), h5_fn=h5_fn, IR_df=only14_manifest)

ix = 21
eg_row = human_exp_df.loc[ix]
target, sr = librosa.load(eg_row.excerpt_src_fn, sr=None)
distractor, sr = librosa.load(eg_row.excerpt_distractor_src_fn, sr=None)

spl = 60
target = util_audio.set_dBSPL(target, spl)
distractor = util_audio.set_dBSPL(distractor, spl)

target_spatial = util_audio.spatialize_sound(target, target_brir)
distractor_spatial = util_audio.spatialize_sound(distractor, dist_brir)

# combine sounds
mixture = target_spatial + distractor_spatial
print(eg_row.sex_cond)
ipd.Audio(mixture.T, rate=sr, normalize=False)

diff


In [14]:
## try writing and re-reading (make sure to write as samples x channels)
out_name = (stim_out_path/f"test_mixture_{ix}.wav").as_posix()
sf.write(out_name, mixture.astype('float32'), sr)
ipd.Audio(out_name, normalize=False)

## Write stimuli manifest 

In [15]:
## write out human df 
output_name = stim_out_path / "human_azim_spotlight_stim_manifest.pdpkl"
# human_exp_df.to_pickle(output_name)

### Write audio to output directory 
attempt in notebook 
Will write every example to every condition - use experiment script to manage randomization   


##### Note: soundfile writes binaural audio formatted as samples x channels

In [16]:
SR = 44100
dBSPL = 60
# timing in seconds 
trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
mixture_onset = int((isi + signal_dur) * SR) # in frames
sig_len_frames = int(signal_dur * SR)

trial_signal = np.zeros((int(SR * trial_dur), 2), dtype=np.float32)

trial_signal[:len(target_spatial), :] = target_spatial
trial_signal[mixture_onset:, :] = mixture

# ipd.Audio(trial_signal.T, rate=SR, normalize=False)



## Demo write audio 

audio written in `get_swc_binaural_azi_spotlight_stim_2024.py`    

Like mono experiment, will save audio as:

- sounds/  
    - condition_00/  
        - f_000.wav  
        - m_000.wav  
        - ...  
    - condition_01/  
        - ...  
    - ...


filenames will have structure condition_<cond_ix>/<target_sex_initial>_<exmpt_word_key_ix>.wav

In [17]:
from tqdm import tqdm 

In [18]:
#### Init audio params 
SR = 44100
dBSPL = 60
# timing in seconds 
trial_dur = 4.5
signal_dur = 2
isi = 0.5
win_dur = 10 # ms 
mixture_onset = int((isi + signal_dur) * SR) # in frames
sig_len_frames = int(signal_dur * SR)
elevation = 0 

# compute level for target and distractor such that sum is 60 dBSPL
# dB_indiv = dBSPL - 10 * np.log10(2)  # 2 here for 2 signals, scaling is generally 10 * np.log10(n_signals)

# iterate over conditions:
for cond_ix, cond_dict in condition_dict.items():
    target_azim, distractor_delta, distractor_azim = cond_dict.values()
    out_dir = stim_out_path / "sounds" / f"condition_{cond_ix:02}"
    out_dir.mkdir(exist_ok=True, parents=True)
    
    # init brirs 
    target_brir = get_brir(coords=(target_azim, elevation), h5_fn=h5_fn, IR_df=only14_manifest)
    dist_brir = get_brir(coords=(distractor_azim, elevation), h5_fn=h5_fn, IR_df=only14_manifest)
    # iterate over trials
    for row in tqdm(human_exp_df.itertuples(), total=len(human_exp_df), desc=f"Making stimuli for {out_dir.stem}: {target_azim} target azim vs {distractor_azim} distractor azim"):
        trial_audio = np.zeros((int(SR * trial_dur), 2), dtype=np.float32) 
        cue_wav, _ = librosa.load(row.excerpt_cue_src_fn, sr=SR)
        target_wav, _ = librosa.load(row.excerpt_src_fn, sr=SR)
        distractor_wav, _ = librosa.load(row.excerpt_distractor_src_fn, sr=SR)

        # set levels for each signal 
        cue_wav = util_audio.set_dBSPL(cue_wav, dBSPL)
        target_wav = util_audio.set_dBSPL(target_wav, dBSPL)
        distractor_wav = util_audio.set_dBSPL(distractor_wav, dBSPL)

        # spatialize 
        cue_spatial = util_audio.spatialize_sound(cue_wav, target_brir)
        target_spatial = util_audio.spatialize_sound(target_wav, target_brir)
        distractor_spatial = util_audio.spatialize_sound(distractor_wav, dist_brir)

        # combine target and distractor
        mixture = target_spatial + distractor_spatial

        # add to trial audio_array 
        trial_audio[:len(cue_spatial), :] = cue_spatial
        trial_audio[mixture_onset:, :] = mixture

        # setup naming and write
        targ_sex_initial = row.gender[0]
        exp_word_ix = row.exp_word_ix
        out_name = out_dir / f'{targ_sex_initial}_{exp_word_ix:03}.wav'
        # write audio 
        sf.write(out_name, trial_audio.astype('float32'), samplerate=SR)




Making stimuli for condition_00: 0 target azim vs 270 distractor azim:   0%|          | 0/976 [00:00<?, ?it/s]

Making stimuli for condition_00: 0 target azim vs 270 distractor azim: 100%|██████████| 976/976 [01:01<00:00, 15.85it/s]
Making stimuli for condition_01: 0 target azim vs 300 distractor azim: 100%|██████████| 976/976 [01:00<00:00, 16.09it/s]
Making stimuli for condition_02: 0 target azim vs 320 distractor azim: 100%|██████████| 976/976 [01:00<00:00, 16.05it/s]
Making stimuli for condition_03: 0 target azim vs 330 distractor azim: 100%|██████████| 976/976 [01:00<00:00, 16.07it/s]
Making stimuli for condition_04: 0 target azim vs 340 distractor azim: 100%|██████████| 976/976 [01:01<00:00, 15.93it/s]
Making stimuli for condition_05: 0 target azim vs 350 distractor azim: 100%|██████████| 976/976 [01:01<00:00, 15.97it/s]
Making stimuli for condition_06: 0 target azim vs 0 distractor azim: 100%|██████████| 976/976 [01:01<00:00, 15.99it/s]
Making stimuli for condition_07: 0 target azim vs 10 distractor azim: 100%|██████████| 976/976 [01:00<00:00, 16.05it/s]
Making stimuli for condition_08: 0 

'dist_azim_delta'